# ARGO Profile Preprocessing — Pacific/Indian/Atlantic Ocean

This notebook takes the raw parquet file with ARGO profiles from the choosen Ocean, applies filters, aggregates vertical statistics per profile, and generates the train / val / test splits that feed the anomaly detection model  (RF, XGBoost, etc).

**Input example:** `indian_ocean_clean4.parquet` (observation-level measurements: pressure, temperature, salinity and their QC flags).

**Output:** a parquet of clean profiles, one of filtered observations, and three parquets (`train` / `val` / `test`) with features ready for training.

### Contents
1. Configuration and paths
2. Execution parameters
3. Quality control functions
4. Vertical feature engineering
5. Stage 1 — chunked loading and cleaning
6. Stage 2 — building the profile table
7. Undersampling
8. Split strategies (temporal / by float)
9. Saving results
10. Pipeline execution
11. Final summary


## 1. Configuration and paths

We define the input/output paths and the constants used throughout the pipeline: chunk size for reading, minimum number of levels to accept a profile, and the dates that separate train / val / test when using the temporal split.


#Correct: add a 

In [1]:
import argparse
import gc
import os
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import joblib

warnings.filterwarnings("ignore")

In [2]:
# Dataset paths
PARQUET_PATH = "/work/drgarcia/Dataset/indian_ocean/2018-2022/1.processed/indian_2018_2022_clean2.parquet"
OUTPUT_BASE  = "/work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022"

# Pipeline constants
CHUNK_SIZE = 500_000          # rows read per batch from the parquet
MIN_LEVELS = 10                # minimum pressure levels to accept a profile
DATE_COL   = "JULD"
TRAIN_END  = pd.Timestamp("2021-01-01")#train 2018-2019-2020
VAL_END    = pd.Timestamp("2022-01-01") #val 2021, test 2022

# Columns read from the raw parquet
VARS_INTERES = [
    'DATA_TYPE', 'REFERENCE_DATE_TIME', 'DATE_CREATION', 'DATE_UPDATE',
    'PLATFORM_NUMBER', 'CYCLE_NUMBER', 'DIRECTION', 'DATA_CENTRE', 'DATA_MODE',
    'PLATFORM_TYPE', 'JULD', 'JULD_QC', 'JULD_LOCATION',
    'LATITUDE', 'LONGITUDE', 'POSITION_QC', 'PROFILE_PRES_QC',
    'PROFILE_TEMP_QC', 'PROFILE_PSAL_QC',
    'PRES', 'PRES_QC', 'PRES_ADJUSTED', 'PRES_ADJUSTED_QC', 'PRES_ADJUSTED_ERROR',
    'TEMP', 'TEMP_QC', 'TEMP_ADJUSTED', 'TEMP_ADJUSTED_QC', 'TEMP_ADJUSTED_ERROR',
    'PSAL', 'PSAL_QC', 'PSAL_ADJUSTED', 'PSAL_ADJUSTED_QC', 'PSAL_ADJUSTED_ERROR',
]

REQUIRED_COLS = [
    "PLATFORM_NUMBER", "CYCLE_NUMBER", "DIRECTION", "PRES", "TEMP", "PSAL",
    "PRES_QC", "TEMP_QC", "PSAL_QC", "JULD_QC", "JULD",
]

VALID_OBS_QC  = {1, 2, 3, 4}   # accepted QC flags at observation level
VALID_JULD_QC = {1, 2}         # accepted QC flags for the profile date
VALID_POSITION_QC = {1, 2}    # accepted QC flags for the profile position (lat/lon)

# Profile key: the same (float, cycle) can have an ascending profile
# and a descending one. These are two physically distinct measurements,
# taken at different times, so DIRECTION is part of the key to avoid
# mixing their observations into a single "Frankenstein" profile with
# non-monotonic pressure.
PROFILE_KEY = ["PLATFORM_NUMBER", "CYCLE_NUMBER", "DIRECTION"]

# Metadata columns that travel alongside each profile but are NOT features
META_COLS = [
    "PLATFORM_NUMBER", "CYCLE_NUMBER", "is_bad", "PRES_is_bad", "TEMP_is_bad",
    "PSAL_is_bad", "JULD_ns", "date", 'LATITUDE', 'LONGITUDE', "DIRECTION",
    "DATA_CENTRE", "DATA_MODE",  # DATA_MODE: R=Real-Time, A=Adjusted, D=Delayed-Mode
    'PROFILE_PRES_QC', 'PROFILE_TEMP_QC', 'PROFILE_PSAL_QC',
    'DATE_CREATION', 'DATE_UPDATE',
]

OBS_QC_COLS = [
    "PRES_QC", "TEMP_QC", "PSAL_QC", "JULD_QC", "POSITION_QC",
    "PRES_ADJUSTED_QC", "TEMP_ADJUSTED_QC", "PSAL_ADJUSTED_QC",
]
PROFILE_QC_COLS = ["PROFILE_PRES_QC", "PROFILE_TEMP_QC", "PROFILE_PSAL_QC"]


## 2. Execution parameters

The original script was run from the terminal using `argparse` (`--split`, `--ascending_only`, `--undersample`). In the notebook we replace those with simple variables you can change directly in the cell below.

- `SPLIT_MODE`: `"time"` separates train/val/test by date; `"platform"` separates by float (avoids leakage between splits when the same float appears in both).
- `ASCENDING_ONLY`: if `True`, descending profiles are discarded.
- `UNDERSAMPLE`: if `True`, balances the majority class in the training set.


In [3]:
SPLIT_MODE      = "time"     # "time" or "platform"
ASCENDING_ONLY  = True
UNDERSAMPLE     = False

# DATA_MODE: R=Real-Time, A=Adjusted, D=Delayed-Mode
# If True, keep only delayed-mode ('D') profiles/observations.
# If False, no filtering is applied on DATA_MODE.
DELAYED_MODE_ONLY = True

# POSITION_QC: quality flag of the profile position (LATITUDE/LONGITUDE).
# If True, keep only rows with POSITION_QC in {1, 2} (NaN is always kept).
POSITION_QC_FILTER = True

filter_juld_qc = (SPLIT_MODE == "time")

desc_tag   = "ascA" if ASCENDING_ONLY else "ascAD"
under_tag  = "_under" if UNDERSAMPLE else ""
dmode_tag  = "_dmodeD" if DELAYED_MODE_ONLY else ""
OUTPUT_DIR = os.path.join(OUTPUT_BASE, f"split_{SPLIT_MODE}_{desc_tag}{under_tag}{dmode_tag}")

print("Split           :", SPLIT_MODE)
print("Ascending only  :", ASCENDING_ONLY)
print("Undersample     :", UNDERSAMPLE)
print("Filter JULD QC  :", filter_juld_qc)
print("Delayed mode only:", DELAYED_MODE_ONLY)
print("Position QC filter:", POSITION_QC_FILTER)
print("Output dir      :", OUTPUT_DIR)


Split           : time
Ascending only  : True
Undersample     : False
Filter JULD QC  : True
Delayed mode only: True
Position QC filter: True
Output dir      : /work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022/split_time_ascA_dmodeD


## 3. Quality control functions

ARGO QC flags arrive as strings/objects; we cast them to a numeric type (`Int8`, which supports NaN) so they can be filtered and compared consistently across all chunks.


In [4]:
def cast_qc_columns(chunk: pd.DataFrame) -> pd.DataFrame:
    """Casts QC flags to a consistent type so they can be filtered and compared."""
    for col in OBS_QC_COLS:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype("Int8")
    for col in PROFILE_QC_COLS:
        if col in chunk.columns:
            chunk[col] = chunk[col].astype(str).replace("nan", pd.NA)
    return chunk


### Global filter diagnostics

`GlobalDiag` accumulates, chunk by chunk, the distribution of each QC flag and the effect of each filter on the entire dataset. This prevents the final report from being biased by a sample from a single chunk (the first 500k records don't necessarily represent the whole file).


In [5]:
class GlobalDiag:
    """Accumulates filtering statistics across all chunks,
    so the effect of each filter can be reported over the entire file
    instead of over a partial sample."""

    def __init__(self):
        self.n_raw = 0
        self.qc_value_counts = {col: defaultdict(int)
                                 for col in ["JULD_QC", "TEMP_QC", "PSAL_QC", "PRES_QC", "POSITION_QC"]}
        self.direction_counts = defaultdict(int)
        self.dmode_counts = defaultdict(int)
        self.nan_counts = defaultdict(int)
        self.steps = {
            "start":            0,
            "juld_qc_filter":   0,
            "obs_qc_filter":    0,
            "obs_qc_temp_only": 0,
            "obs_qc_psal_only": 0,
            "obs_qc_pres_only": 0,
            "dropna":           0,
            "direction_filter": 0,
            "position_qc_filter": 0,
            "dmode_filter":     0,
        }

    def update(self, chunk_raw: pd.DataFrame, filter_juld_qc: bool, ascending_only: bool,
               delayed_mode_only: bool = True, position_qc_filter: bool = True):
        chunk = cast_qc_columns(chunk_raw.copy())
        n0 = len(chunk)
        self.n_raw += n0
        self.steps["start"] += n0

        for col in ["JULD_QC", "TEMP_QC", "PSAL_QC", "PRES_QC", "POSITION_QC"]:
            if col in chunk.columns:
                vc = chunk[col].value_counts(dropna=False)
                for val, cnt in vc.items():
                    self.qc_value_counts[col][val] += int(cnt)

        if "DIRECTION" in chunk.columns:
            vc = chunk["DIRECTION"].value_counts(dropna=False)
            for val, cnt in vc.items():
                self.direction_counts[val] += int(cnt)

        if "DATA_MODE" in chunk.columns:
            vc = chunk["DATA_MODE"].value_counts(dropna=False)
            for val, cnt in vc.items():
                self.dmode_counts[val] += int(cnt)

        for col in ["PRES", "TEMP", "PSAL", DATE_COL]:
            if col in chunk.columns:
                self.nan_counts[col] += int(chunk[col].isna().sum())

        remaining = chunk
        if filter_juld_qc:
            remaining = remaining[remaining["JULD_QC"].isin(VALID_JULD_QC)]
        self.steps["juld_qc_filter"] += len(remaining)

        after = remaining[
            remaining["TEMP_QC"].isin(VALID_OBS_QC) &
            remaining["PSAL_QC"].isin(VALID_OBS_QC) &
            remaining["PRES_QC"].isin(VALID_OBS_QC)
        ]
        self.steps["obs_qc_filter"] += len(after)
        self.steps["obs_qc_temp_only"] += (~remaining["TEMP_QC"].isin(VALID_OBS_QC)).sum()
        self.steps["obs_qc_psal_only"] += (~remaining["PSAL_QC"].isin(VALID_OBS_QC)).sum()
        self.steps["obs_qc_pres_only"] += (~remaining["PRES_QC"].isin(VALID_OBS_QC)).sum()
        remaining = after

        if position_qc_filter and "POSITION_QC" in remaining.columns:
            remaining = remaining[remaining["POSITION_QC"].isin(VALID_POSITION_QC)]
        self.steps["position_qc_filter"] += len(remaining)

        after = remaining.dropna(subset=["JULD", "PRES", "TEMP", "PSAL", "LATITUDE", "LONGITUDE"]) #executed always, without mattering if position_qc true
        #dropna_cols = ["JULD", "PRES", "TEMP", "PSAL"]
        #if position_qc_filter:
        #    dropna_cols += ["LATITUDE", "LONGITUDE"]
        #after = remaining.dropna(subset=dropna_cols)
        self.steps["dropna"] += len(after)
        remaining = after

        if ascending_only:
            remaining = remaining[remaining["DIRECTION"] == "A"]
        self.steps["direction_filter"] += len(remaining)

        if delayed_mode_only and "DATA_MODE" in remaining.columns:
            remaining = remaining[remaining["DATA_MODE"] == "D"]
        self.steps["dmode_filter"] += len(remaining)

    def report(self, filter_juld_qc: bool, ascending_only: bool, delayed_mode_only: bool = True,
               position_qc_filter: bool = True):
        n = self.n_raw
        print()
        print("=" * 70)
        print(f"  GLOBAL FILTER DIAGNOSTICS (full file, {n:,} raw rows)")
        print("=" * 70)

        for col, counts in self.qc_value_counts.items():
            print(f"\n  {col} distribution:")
            for val in sorted(counts, key=lambda x: (x is pd.NA, str(x))):
                cnt = counts[val]
                pct = cnt / n * 100 if n else 0
                valid_set = {1, 2} if col in ("JULD_QC", "POSITION_QC") else {1, 2, 3, 4}
                marker = " <- VALID" if val in valid_set else ""
                print(f"    {str(val):>6}  {cnt:>12,}  ({pct:5.1f}%){marker}")

        print("\n  DIRECTION distribution:")
        for val, cnt in self.direction_counts.items():
            marker = " <- KEPT" if (not ascending_only or val == "A") else " <- DISCARDED"
            pct = cnt / n * 100 if n else 0
            print(f"    {str(val):>6}  {cnt:>12,}  ({pct:5.1f}%){marker}")

        print("\n  DATA_MODE distribution:")
        for val, cnt in self.dmode_counts.items():
            marker = " <- KEPT" if (not delayed_mode_only or val == "D") else " <- DISCARDED"
            pct = cnt / n * 100 if n else 0
            print(f"    {str(val):>6}  {cnt:>12,}  ({pct:5.1f}%){marker}")

        print("\n  NaN in key columns:")
        for col, cnt in self.nan_counts.items():
            pct = cnt / n * 100 if n else 0
            print(f"    {col:20s}  {cnt:>12,} NaN  ({pct:5.1f}%)")

        print("\n  Impact of each filter (full file):")
        prev = n
        print(f"    {'start':30s}  {self.steps['start']:>12,}  (100.0%)")
        prev = self.steps["start"]

        if filter_juld_qc:
            cur = self.steps["juld_qc_filter"]
            print(f"    {'JULD_QC filter':30s}  {cur:>12,}  "
                  f"({cur / n * 100:5.1f}%)  dropped {prev - cur:,}")
            prev = cur

        cur = self.steps["obs_qc_filter"]
        print(f"    {'obs QC filter (T+S+P)':30s}  {cur:>12,}  "
              f"({cur / n * 100:5.1f}%)  dropped {prev - cur:,}")
        print(f"      -> TEMP_QC alone  {self.steps['obs_qc_temp_only']:>12,} rows would be lost on their own")
        print(f"      -> PSAL_QC alone  {self.steps['obs_qc_psal_only']:>12,} rows would be lost on their own")
        print(f"      -> PRES_QC alone  {self.steps['obs_qc_pres_only']:>12,} rows would be lost on their own")
        prev = cur

        if position_qc_filter:
            cur = self.steps["position_qc_filter"]
            print(f"    {'POSITION_QC filter':30s}  {cur:>12,}  "
                  f"({cur / n * 100:5.1f}%)  dropped {prev - cur:,}")
            prev = cur

        cur = self.steps["dropna"]
        print(f"    {'explicit dropna':30s}  {cur:>12,}  "
              f"({cur / n * 100:5.1f}%)  dropped {prev - cur:,}")
        prev = cur

        if ascending_only:
            cur = self.steps["direction_filter"]
            print(f"    {'DIRECTION == A':30s}  {cur:>12,}  "
                  f"({cur / n * 100:5.1f}%)  dropped {prev - cur:,}")
            prev = cur

        if delayed_mode_only:
            cur = self.steps["dmode_filter"]
            print(f"    {'DATA_MODE == D':30s}  {cur:>12,}  "
                  f"({cur / n * 100:5.1f}%)  dropped {prev - cur:,}")
            prev = cur

        print(f"\n  -> Rows surviving all filters: {prev:,} / {n:,} ({prev / n * 100:.1f}%)")
        print("=" * 70)
        print()


## 4. Vertical feature engineering

For each profile we compute descriptive statistics of temperature, salinity and pressure: max, min, mean, std, range, and statistics of the vertical gradient (difference between consecutive levels). These are the features the model will see.


In [6]:
def _feats(arr: np.ndarray, prefix: str) -> dict:
    """Vertical statistics of a profile (max, min, mean, std, range and gradients)."""
    diffs = np.abs(np.diff(arr))
    return {
        f"{prefix}_max":       float(arr.max()),
        f"{prefix}_min":       float(arr.min()),
        f"{prefix}_mean":      float(arr.mean()),
        f"{prefix}_std":       float(arr.std()),
        f"{prefix}_range":     float(arr.max() - arr.min()),
        f"{prefix}_max_grad":  float(diffs.max())  if len(diffs) else 0.0,
        f"{prefix}_mean_grad": float(diffs.mean()) if len(diffs) else 0.0,
        f"{prefix}_grad_std":  float(diffs.std())  if len(diffs) else 0.0,
    }


## 5. Stage 1 — Chunked loading and cleaning

The parquet is read in batches of `CHUNK_SIZE` rows (it doesn't comfortably fit in memory all at once). For each chunk:

1. Global QC diagnostics are recorded.
2. QC filters (date, observation) and direction filters are applied.
3. Surviving observations are grouped by `(float, cycle, direction)` and their T/S/P arrays are accumulated for the feature-engineering stage.


In [7]:
def load_and_accumulate(ascending_only: bool, filter_juld_qc: bool, delayed_mode_only: bool = True,
                         position_qc_filter: bool = True):
    print("-" * 70)
    print("Loading and cleaning (by chunks)")
    print("-" * 70)

    pf = pq.ParquetFile(PARQUET_PATH)

    available_cols = pq.read_schema(PARQUET_PATH).names
    cols_to_read   = [c for c in VARS_INTERES if c in available_cols]
    missing        = set(VARS_INTERES) - set(cols_to_read)
    if missing:
        print("\n  [WARNING] Columns not found in the parquet:")
        for m in sorted(missing):
            print(f"         - {m}")

    missing_required = set(REQUIRED_COLS) - set(cols_to_read)
    assert not missing_required, f"Missing required columns: {missing_required}"

    if delayed_mode_only and "DATA_MODE" not in cols_to_read:
        raise ValueError(
            "DELAYED_MODE_ONLY=True but 'DATA_MODE' column is not available in the parquet."
        )

    meta      = pq.read_metadata(PARQUET_PATH)
    total_raw = meta.num_rows
    print(f"\n  Raw rows in the parquet     : {total_raw:,}")
    print(f"  Columns to read             : {len(cols_to_read)} / {len(VARS_INTERES)}")

    diag = GlobalDiag()

    filter_stats = {
        "n_raw":         0,
        "after_juld":    0,
        "after_obs_qc":  0,
        "after_position_qc": 0,
        "after_dropna":  0,
        "after_dir":     0,
        "after_dmode":   0,
    }

    # stats is indexed by (PLATFORM_NUMBER, CYCLE_NUMBER, DIRECTION)
    stats = defaultdict(lambda: {
        "T_vals": [], "S_vals": [], "P_vals": [], "JULD_ns": [],
        "T_bad": 0,   "S_bad": 0,  "P_bad": 0,
    })
    obs_chunks        = []
    total_rows_read   = 0
    rows_after_filter = 0

    for i, batch in enumerate(pf.iter_batches(batch_size=CHUNK_SIZE, columns=cols_to_read)):
        chunk = batch.to_pandas()
        n_raw = len(chunk)
        total_rows_read += n_raw
        filter_stats["n_raw"] += n_raw

        diag.update(chunk.copy(), filter_juld_qc, ascending_only, delayed_mode_only, position_qc_filter)

        # Filter 1: cast QC columns
        chunk = cast_qc_columns(chunk)

        # Filter 2: JULD temporal quality (only applies to the temporal split)
        if filter_juld_qc:
            chunk = chunk[chunk["JULD_QC"].isin(VALID_JULD_QC)]
        filter_stats["after_juld"] += len(chunk)

        # Filter 3: observation QC, flags 1/2/3/4 accepted; NA and 9 (missing) discarded
        chunk = chunk[
            chunk["TEMP_QC"].isin(VALID_OBS_QC) &
            chunk["PSAL_QC"].isin(VALID_OBS_QC) &
            chunk["PRES_QC"].isin(VALID_OBS_QC)
        ]
        filter_stats["after_obs_qc"] += len(chunk)

        # Filter 4: POSITION_QC, only flags 1/2 accepted
        if position_qc_filter and "POSITION_QC" in chunk.columns:
            chunk = chunk[chunk["POSITION_QC"].isin(VALID_POSITION_QC)]
        filter_stats["after_position_qc"] += len(chunk)

        # Filter 5: explicit dropna, normally doesn't discard anything new
        nan_before = len(chunk)
        for col in ["JULD", "PRES", "TEMP", "PSAL", "LATITUDE", "LONGITUDE"]:
            if col in chunk.columns:
                chunk = chunk[chunk[col].notna()]
        filter_stats["after_dropna"] += len(chunk)
        nan_dropped = nan_before - len(chunk)
        if nan_dropped > 0:
            print(f"  [WARNING] Chunk {i + 1}: explicit dropna removed {nan_dropped:,} "
                  f"rows not caught by the QC filter — check source data")

        # Filter 6: direction, only ascending (A) is kept if applicable
        if ascending_only:
            chunk = chunk[chunk["DIRECTION"] == "A"]
        filter_stats["after_dir"] += len(chunk)

        # Filter 7: DATA_MODE, only delayed mode ('D') is kept if applicable
        if delayed_mode_only:
            chunk = chunk[chunk["DATA_MODE"] == "D"]
        filter_stats["after_dmode"] += len(chunk)

        if chunk.empty:
            del chunk
            gc.collect()
            continue

        rows_after_filter += len(chunk)
        chunk.sort_values(PROFILE_KEY + ["PRES"], inplace=True)
        obs_chunks.append(chunk)

        for (plat, cyc, direc), grp in chunk.groupby(PROFILE_KEY, sort=False):
            key = (plat, cyc, direc)
            s   = stats[key]
            s["T_vals"].append(grp["TEMP"].values.astype(np.float32))
            s["S_vals"].append(grp["PSAL"].values.astype(np.float32))
            s["P_vals"].append(grp["PRES"].values.astype(np.float32))
            s["JULD_ns"].append(
                grp[DATE_COL].values.astype("datetime64[ns]").astype(np.int64)
            )
            if not s["T_bad"] and grp["TEMP_QC"].isin([3, 4]).any():
                s["T_bad"] = 1
            if not s["S_bad"] and grp["PSAL_QC"].isin([3, 4]).any():
                s["S_bad"] = 1
            if not s["P_bad"] and grp["PRES_QC"].isin([3, 4]).any():
                s["P_bad"] = 1

        del chunk
        gc.collect()
        print(f"  Chunk {i + 1:3d} — {total_rows_read:,} raw | {rows_after_filter:,} kept")

    n = filter_stats["n_raw"]
    print()
    print("-" * 70)
    print("  GLOBAL FILTER SUMMARY")
    print("-" * 70)
    steps = [
        ("Raw rows",                    filter_stats["n_raw"]),
        ("After JULD_QC filter",        filter_stats["after_juld"]   if filter_juld_qc else filter_stats["n_raw"]),
        ("After obs QC filter",         filter_stats["after_obs_qc"]),
        ("After POSITION_QC filter",    filter_stats["after_position_qc"] if position_qc_filter else filter_stats["after_obs_qc"]),
        ("After dropna",                filter_stats["after_dropna"]),
        ("After direction filter",      filter_stats["after_dir"]    if ascending_only else filter_stats["after_dropna"]),
        ("After DATA_MODE=D filter",    filter_stats["after_dmode"]  if delayed_mode_only else filter_stats["after_dir"]),
    ]
    prev = n
    for label, count in steps:
        dropped     = prev - count
        pct_kept    = count / n * 100
        pct_dropped = dropped / prev * 100 if prev > 0 else 0
        print(f"  {label:35s}  {count:>12,}  ({pct_kept:5.1f}% of total)  "
              f"[-{dropped:,} = -{pct_dropped:.1f}% of previous step]")
        prev = count
    print("-" * 70)
    print(f"  Rows remaining after DATA_MODE filter (delayed_mode_only={delayed_mode_only}) : "
          f"{filter_stats['after_dmode'] if delayed_mode_only else filter_stats['after_dir']:,}")
    print("-" * 70)

    diag.report(filter_juld_qc, ascending_only, delayed_mode_only, position_qc_filter)

    print(f"\n  Unique keys (float, cycle, direction) : {len(stats):,}")
    print("\n  Consolidating clean observations...")
    df_obs = pd.concat(obs_chunks, ignore_index=True)
    del obs_chunks
    gc.collect()
    print(f"  df_obs shape : {df_obs.shape}")

    return stats, total_rows_read, rows_after_filter, df_obs


## 6. Stage 2 — Building the profile table

For each profile, observations are sorted by pressure, vertical features are computed, and a row is built per profile. A profile is discarded only if it has fewer than `MIN_LEVELS` levels.

If `JULD` is not unique within a profile (something that in theory shouldn't happen now that we also group by direction), the profile is not discarded: it's simply recorded in the statistics and the first available `JULD` value is used.


In [8]:
def build_profile_df(stats: dict, df_obs: pd.DataFrame):
    print()
    print("-" * 70)
    print("Vertical feature engineering")
    print("-" * 70)

    rows              = []
    skipped_levels    = 0
    n_non_unique_juld = 0   # diagnostic only, profile is not discarded for this

    for key, s in stats.items():
        T = np.concatenate(s["T_vals"])
        S = np.concatenate(s["S_vals"])
        P = np.concatenate(s["P_vals"])
        J = np.concatenate(s["JULD_ns"])

        order   = np.argsort(P)
        T, S, P = T[order], S[order], P[order]

        # dedup igual que el CNN, para contar niveles ÚNICOS de presión
        uP, inv = np.unique(P, return_inverse=True)
        if len(uP) < len(P):
            T = np.bincount(inv, weights=T) / np.bincount(inv)
            S = np.bincount(inv, weights=S) / np.bincount(inv)
            P = uP

        if len(P) < MIN_LEVELS:
            skipped_levels += 1
            continue

        # JULD_ns is not reordered by time when sorting by pressure, it follows
        # the arrival order of the chunks; if there is more than one unique
        # value we record it as a diagnostic and use J[0].
        unique_j = np.unique(J)
        if len(unique_j) > 1:
            n_non_unique_juld += 1

        row = {"PLATFORM_NUMBER": key[0], "CYCLE_NUMBER": key[1], "DIRECTION": key[2]}
        row.update(_feats(T, "TEMP"))
        row.update(_feats(S, "PSAL"))
        row.update(_feats(P, "PRES"))

        row["TEMP_is_bad"] = s["T_bad"]
        row["PSAL_is_bad"] = s["S_bad"]
        row["PRES_is_bad"] = s["P_bad"]
        row["is_bad"]      = int(s["T_bad"] or s["S_bad"] or s["P_bad"])
        row["JULD_ns"]     = int(J[0])
        rows.append(row)

    df = pd.DataFrame(rows)
    del rows, stats
    gc.collect()

    df["date"] = pd.to_datetime(df["JULD_ns"].astype(np.int64), unit="ns")

    # The first metadata record per profile is used; grouping also by
    # DIRECTION already removes the most common cause of apparent "duplicates".
    extra_meta_cols = [
        "PLATFORM_NUMBER", "CYCLE_NUMBER", "DIRECTION", "DATA_CENTRE", "DATA_MODE",
        "PROFILE_PRES_QC", "PROFILE_TEMP_QC", "PROFILE_PSAL_QC",
        "DATE_CREATION", "DATE_UPDATE", "LATITUDE", "LONGITUDE"
    ]
    cols_to_extract = [c for c in extra_meta_cols if c in df_obs.columns]

    meta_extraido = (
        df_obs[cols_to_extract]
        .groupby(PROFILE_KEY, sort=False)
        .first()
        .reset_index()
    )

    df = df.merge(meta_extraido, on=PROFILE_KEY, how="left")

    print(f"  Profiles discarded (< {MIN_LEVELS} levels) : {skipped_levels:,}")
    print(f"  Profiles with non-unique JULD        : {n_non_unique_juld:,}  "
          f"<- diagnostic only, not discarded (JULD_ns of the first observation is used)")
    print(f"  Valid profiles                       : {len(df):,}")
    print(f"  Anomalous profiles                   : {df['is_bad'].sum():,} "
          f"({df['is_bad'].mean():.2%})")
    print(f"  Date range                           : "
          f"{df['date'].min().date()} -> {df['date'].max().date()}")
    print(f"  TEMP anomalies : {df['TEMP_is_bad'].sum():,}")
    print(f"  PSAL anomalies : {df['PSAL_is_bad'].sum():,}")
    print(f"  PRES anomalies : {df['PRES_is_bad'].sum():,}")

    return df, df_obs


## 7. Undersampling (optional, train only)

When the anomalous profile class is heavily imbalanced, this function undersamples the majority class in the training set to bring the class ratio closer together. It is never applied to validation or test.


In [9]:
def undersample_train(X_train, y_train, ratio=1, seed=42):
    idx_bad  = y_train[y_train == 1].index
    idx_good = y_train[y_train == 0].index
    n_keep   = min(len(idx_good), len(idx_bad) * ratio)
    rng      = np.random.default_rng(seed)
    idx_good_sampled = rng.choice(idx_good, size=n_keep, replace=False)
    idx_keep = np.concatenate([idx_bad, idx_good_sampled])
    X_train  = X_train.loc[idx_keep]
    y_train  = y_train.loc[idx_keep]
    print(f"  [Undersample] kept {len(idx_keep):,} profiles "
          f"(anomalous={len(idx_bad):,}, normal={n_keep:,})")
    return X_train, y_train


## 8. Split strategies

**Temporal split:** train / val / test are separated by date (`TRAIN_END`, `VAL_END`). This is the most realistic scenario for a model that will run on future data, but the same float can appear in more than one split.

**Split by float:** floats are randomly distributed across train/val/test (70/15/15). This prevents the model from seeing the same float in train and test, at the cost of not respecting temporal order.


In [10]:
def split_temporal(df, feature_cols, do_undersample):
    print()
    print("-" * 70)
    print("Temporal split")
    print("-" * 70)

    train_mask = df["date"] < TRAIN_END
    val_mask   = (df["date"] >= TRAIN_END) & (df["date"] < VAL_END)
    test_mask  = df["date"] >= VAL_END

    if not (train_mask.any() and val_mask.any() and test_mask.any()):
        raise ValueError("One of the splits ended up empty — check TRAIN_END/VAL_END against the date range.")

    def _split_info(mask, name):
        sub         = df.loc[mask]
        plats_train = set(df.loc[train_mask, "PLATFORM_NUMBER"])
        plats_sub   = set(sub["PLATFORM_NUMBER"])
        seen        = plats_sub & plats_train if name != "train" else set()
        print(f"  {name:6s} : {mask.sum():>7,} profiles | "
              f"{sub['date'].min().date()} -> {sub['date'].max().date()} | "
              f"{sub['is_bad'].mean():.2%} anomalous | "
              f"{sub['PLATFORM_NUMBER'].nunique():,} unique floats"
              + (f" | {len(seen):,} already seen in train" if seen else ""))

    _split_info(train_mask, "train")
    _split_info(val_mask,   "val")
    _split_info(test_mask,  "test")

    X_train = df.loc[train_mask, feature_cols].astype(np.float32)
    y_train = df.loc[train_mask, "is_bad"]
    X_val   = df.loc[val_mask,   feature_cols].astype(np.float32)
    y_val   = df.loc[val_mask,   "is_bad"]
    X_test  = df.loc[test_mask,  feature_cols].astype(np.float32)
    y_test  = df.loc[test_mask,  "is_bad"]

    df_test_meta = df.loc[test_mask, META_COLS].copy()

    if do_undersample:
        X_train, y_train = undersample_train(X_train, y_train)
    else:
        print("  Undersample disabled")

    return X_train, y_train, X_val, y_val, X_test, y_test, df_test_meta


In [11]:
def split_platform(df, feature_cols, do_undersample, seed=42):
    print()
    print("=" * 70)
    print("Split by float")
    print("=" * 70)

    platforms = df["PLATFORM_NUMBER"].unique()
    rng       = np.random.default_rng(seed)
    rng.shuffle(platforms)

    n       = len(platforms)
    n_train = int(n * 0.70)
    n_val   = int(n * 0.15)

    train_plats = set(platforms[:n_train])
    val_plats   = set(platforms[n_train: n_train + n_val])
    test_plats  = set(platforms[n_train + n_val:])

    train_mask = df["PLATFORM_NUMBER"].isin(train_plats)
    val_mask   = df["PLATFORM_NUMBER"].isin(val_plats)
    test_mask  = df["PLATFORM_NUMBER"].isin(test_plats)

    assert not (train_plats & val_plats),  "Train/val overlap!"
    assert not (train_plats & test_plats), "Train/test overlap!"
    assert not (val_plats   & test_plats), "Val/test overlap!"

    def _split_info(mask, name, plat_set):
        sub = df.loc[mask]
        print(f"  {name:6s} : {mask.sum():>7,} profiles | "
              f"{len(plat_set):,} unique floats | "
              f"{sub['date'].min().date()} -> {sub['date'].max().date()} | "
              f"{sub['is_bad'].mean():.2%} anomalous")

    _split_info(train_mask, "train", train_plats)
    _split_info(val_mask,   "val",   val_plats)
    _split_info(test_mask,  "test",  test_plats)

    X_train = df.loc[train_mask, feature_cols].astype(np.float32)
    y_train = df.loc[train_mask, "is_bad"]
    X_val   = df.loc[val_mask,   feature_cols].astype(np.float32)
    y_val   = df.loc[val_mask,   "is_bad"]
    X_test  = df.loc[test_mask,  feature_cols].astype(np.float32)
    y_test  = df.loc[test_mask,  "is_bad"]

    df_test_meta = df.loc[test_mask, META_COLS].copy()

    if do_undersample:
        X_train, y_train = undersample_train(X_train, y_train)
    else:
        print("  Undersample disabled")

    return X_train, y_train, X_val, y_val, X_test, y_test, df_test_meta


## 9. Saving results

We save the filtered observations, the clean profile table, the three splits (`train.parquet`, `val.parquet`, `test.parquet`), the test set metadata, and the list of feature columns (needed to reconstruct the same feature order when training or serving the model).


In [12]:
def save_outputs(output_dir, df_clean, df_obs, X_train, y_train,
                  X_val, y_val, X_test, y_test, df_test_meta, feature_cols):

    os.makedirs(output_dir, exist_ok=True)

    obs_path = os.path.join(output_dir, "observations_clean.parquet")
    df_obs.to_parquet(obs_path, index=False, compression="snappy")
    print(f"\n  Observations saved -> {obs_path}  ({len(df_obs):,} rows x {df_obs.shape[1]} cols)")

    clean_path = os.path.join(output_dir, "profiles_clean.parquet")
    df_clean.to_parquet(clean_path, index=False, compression="snappy")
    print(f"  Clean profiles saved -> {clean_path}  ({len(df_clean):,} profiles x {df_clean.shape[1]} cols)")

    for name, X, y in [("train", X_train, y_train),
                        ("val",   X_val,   y_val),
                        ("test",  X_test,  y_test)]:
        split_df           = X.copy()
        split_df["is_bad"] = y.values
        path               = os.path.join(output_dir, f"{name}.parquet")
        split_df.to_parquet(path, index=False, compression="snappy")
        print(f"  Split {name:5s} saved -> {path}  ({len(split_df):,} profiles | {y.mean() * 100:.2f}% anomalous)")

    meta_path = os.path.join(output_dir, "test_meta.parquet")
    df_test_meta.to_parquet(meta_path, index=False, compression="snappy")
    print(f"  Test metadata saved -> {meta_path}")

    feat_path = os.path.join(output_dir, "feature_cols.pkl")
    joblib.dump(feature_cols, feat_path)
    print(f"  Feature columns saved -> {feat_path}  ({len(feature_cols)} features)")


## 10. Pipeline execution

With everything defined, we run both stages and the chosen split. `feature_cols` is computed by excluding everything in `META_COLS`; if at any point you add a new column to the merge in `build_profile_df`, add it to `META_COLS` too so it doesn't leak in as a feature.


In [13]:
stats, total_raw, total_clean, df_obs = load_and_accumulate(
    ascending_only=ASCENDING_ONLY,
    filter_juld_qc=filter_juld_qc,
    delayed_mode_only=DELAYED_MODE_ONLY,
    position_qc_filter=POSITION_QC_FILTER,
)

df_clean, df_obs = build_profile_df(stats, df_obs)

feature_cols = [c for c in df_clean.columns if c not in META_COLS]

print(f"\n  Profiles remaining after filters (delayed_mode_only={DELAYED_MODE_ONLY}) : {len(df_clean):,}")

if SPLIT_MODE == "time":
    X_train, y_train, X_val, y_val, X_test, y_test, df_test_meta = \
        split_temporal(df_clean, feature_cols, UNDERSAMPLE)
else:
    X_train, y_train, X_val, y_val, X_test, y_test, df_test_meta = \
        split_platform(df_clean, feature_cols, UNDERSAMPLE)


----------------------------------------------------------------------
Loading and cleaning (by chunks)
----------------------------------------------------------------------

  Raw rows in the parquet     : 199,049,586
  Columns to read             : 34 / 34


  Chunk   1 — 500,000 raw | 146,937 kept


  Chunk   2 — 1,000,000 raw | 297,114 kept


  Chunk   3 — 1,500,000 raw | 498,442 kept


  Chunk   4 — 2,000,000 raw | 676,636 kept


...

  Chunk 397 — 198,500,000 raw | 82,589,956 kept


  Chunk 398 — 199,000,000 raw | 82,778,005 kept


  Chunk 399 — 199,049,586 raw | 82,791,889 kept

----------------------------------------------------------------------
  GLOBAL FILTER SUMMARY
----------------------------------------------------------------------
  Raw rows                              199,049,586  (100.0% of total)  [-0 = -0.0% of previous step]
  After JULD_QC filter                  198,844,942  ( 99.9% of total)  [-204,644 = -0.1% of previous step]
  After obs QC filter                    92,005,770  ( 46.2% of total)  [-106,839,172 = -53.7% of previous step]
  After POSITION_QC filter               89,057,070  ( 44.7% of total)  [-2,948,700 = -3.2% of previous step]
  After dropna                           89,057,070  ( 44.7% of total)  [-0 = -0.0% of previous step]
  After direction filter                 87,740,574  ( 44.1% of total)  [-1,316,496 = -1.5% of previous step]
  After DATA_MODE=D filter               82,791,889  ( 41.6% of total)  [-4,948,685 = -5.6% of previous step]
------------------------------

  df_obs shape : (82791889, 34)

----------------------------------------------------------------------
Vertical feature engineering
----------------------------------------------------------------------


  Profiles discarded (< 10 levels) : 179
  Profiles with non-unique JULD        : 0  <- diagnostic only, not discarded (JULD_ns of the first observation is used)
  Valid profiles                       : 138,567
  Anomalous profiles                   : 59,708 (43.09%)
  Date range                           : 2018-01-01 -> 2022-12-31
  TEMP anomalies : 42,083
  PSAL anomalies : 59,588
  PRES anomalies : 18,853

  Profiles remaining after filters (delayed_mode_only=True) : 138,567

----------------------------------------------------------------------
Temporal split
----------------------------------------------------------------------
  train  :  88,992 profiles | 2018-01-01 -> 2020-12-31 | 39.10% anomalous | 1,340 unique floats
  val    :  26,795 profiles | 2021-01-01 -> 2021-12-31 | 49.19% anomalous | 838 unique floats | 760 already seen in train
  test   :  22,780 profiles | 2022-01-01 -> 2022-12-31 | 51.48% anomalous | 743 unique floats | 574 already seen in train
  Undersample disab

In [14]:
save_outputs(
    OUTPUT_DIR, df_clean, df_obs,
    X_train, y_train, X_val, y_val, X_test, y_test,
    df_test_meta, feature_cols,
)


  Observations saved -> /work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022/split_time_ascA_dmodeD/observations_clean.parquet  (82,791,889 rows x 34 cols)


  Clean profiles saved -> /work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022/split_time_ascA_dmodeD/profiles_clean.parquet  (138,567 profiles x 42 cols)
  Split train saved -> /work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022/split_time_ascA_dmodeD/train.parquet  (88,992 profiles | 39.10% anomalous)
  Split val   saved -> /work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022/split_time_ascA_dmodeD/val.parquet  (26,795 profiles | 49.19% anomalous)
  Split test  saved -> /work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022/split_time_ascA_dmodeD/test.parquet  (22,780 profiles | 51.48% anomalous)
  Test metadata saved -> /work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022/split_time_ascA_dmodeD/test_meta.parquet
  Feature columns saved -> /work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022/split_time_ascA_dmodeD/feature_cols.pkl  (24 features)


## 11. Final summary


In [15]:
print()
print("-" * 70)
print("Done")
print("-" * 70)
print(f"  Raw rows             : {total_raw:,}")
print(f"  Rows after cleaning  : {total_clean:,}")
print(f"  Rows in df_obs       : {len(df_obs):,}")
print(f"  Clean profiles       : {len(df_clean):,}")
print(f"  Feature columns      : {len(feature_cols)}")
print(f"  Train profiles       : {len(X_train):,}  |  {y_train.mean():.2%} anomalous")
print(f"  Val profiles         : {len(X_val):,}  |  {y_val.mean():.2%} anomalous")
print(f"  Test profiles        : {len(X_test):,}  |  {y_test.mean():.2%} anomalous")
print(f"  Output directory     : {OUTPUT_DIR}")



----------------------------------------------------------------------
Done
----------------------------------------------------------------------
  Raw rows             : 199,049,586
  Rows after cleaning  : 82,791,889
  Rows in df_obs       : 82,791,889
  Clean profiles       : 138,567
  Feature columns      : 24
  Train profiles       : 88,992  |  39.10% anomalous
  Val profiles         : 26,795  |  49.19% anomalous
  Test profiles        : 22,780  |  51.48% anomalous
  Output directory     : /work/drgarcia/Dataset/indian_ocean/2018-2022/2.preprocessed_2018_2022/split_time_ascA_dmodeD
